### Imports

In [1]:
import os
import re
import math
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from collections import Counter, defaultdict
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Primary device : {device}')
if torch.cuda.is_available():
    print(f'GPU count      : {torch.cuda.device_count()}')
    for i in range(torch.cuda.device_count()):
        print(f'  GPU {i}        : {torch.cuda.get_device_name(i)}')

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

Primary device : cuda
GPU count      : 1
  GPU 0        : NVIDIA GeForce RTX 4060 Laptop GPU


### Loading and Inspecting the dataset

In [2]:
DATA_PATH = '/kaggle/input/datasets/abdullah23f3061/customer-support-tickets-csv/customer_support_tickets.csv'
MY_PATH = 'dataset/customer_support_tickets.csv'

df = pd.read_csv(MY_PATH)
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [3]:
df.hist

<bound method hist_frame of       Ticket ID        Customer Name              Customer Email  \
0             1        Marisa Obrien  carrollallison@example.com   
1             2         Jessica Rios    clarkeashley@example.com   
2             3  Christopher Robbins   gonzalestracy@example.com   
3             4     Christina Dillon    bradleyolson@example.org   
4             5    Alexander Carroll     bradleymark@example.com   
...         ...                  ...                         ...   
8464       8465           David Todd          adam28@example.net   
8465       8466           Lori Davis       russell68@example.com   
8466       8467      Michelle Kelley        ashley83@example.org   
8467       8468     Steven Rodriguez         fpowell@example.org   
8468       8469      Steven Davis MD          lori20@example.net   

      Customer Age Customer Gender       Product Purchased Date of Purchase  \
0               32           Other              GoPro Hero       2021-03-22 

In [4]:
df.isna().sum()

Ticket ID                          0
Customer Name                      0
Customer Email                     0
Customer Age                       0
Customer Gender                    0
Product Purchased                  0
Date of Purchase                   0
Ticket Type                        0
Ticket Subject                     0
Ticket Description                 0
Ticket Status                      0
Resolution                      5700
Ticket Priority                    0
Ticket Channel                     0
First Response Time             2819
Time to Resolution              5700
Customer Satisfaction Rating    5700
dtype: int64

### Doing Data cleaning

In [5]:
FOCUS = ['Ticket Description', 'Ticket Subject', 'Ticket Priority', 'Ticket Type', 'Ticket Channel']

df = df[FOCUS].copy()
df.dropna(subset=['Ticket Description', 'Ticket Priority', 'Ticket Channel'], inplace=True)
df.reset_index(drop=True, inplace=True)


In [6]:
df.head()

,Ticket Description,Ticket Subject,Ticket Priority,Ticket Type,Ticket Channel
0,I'm having an issue with the {product_purchase...,Product setup,Critical,Technical issue,Social media
1,I'm having an issue with the {product_purchase...,Peripheral compatibility,Critical,Technical issue,Chat
2,I'm facing a problem with my {product_purchase...,Network problem,Low,Technical issue,Social media
3,I'm having an issue with the {product_purchase...,Account access,Low,Billing inquiry,Social media
4,I'm having an issue with the {product_purchase...,Data loss,Low,Billing inquiry,Email


In [7]:

print(df.shape)
print('\npriority value counts:')
print(df['Ticket Priority'].value_counts())
print('\nchannel value counts:')
print(df['Ticket Channel'].value_counts())

(8469, 5)

priority value counts:
Ticket Priority
Medium      2192
Critical    2129
High        2085
Low         2063
Name: count, dtype: int64

channel value counts:
Ticket Channel
Email           2143
Phone           2132
Social media    2121
Chat            2073
Name: count, dtype: int64


### Label Encoding

In [8]:
class LabelEncoder:
    
    def __init__(self, unknown_index: int = -1):
        self.unknown_index = unknown_index
        self.label2idx: dict = {}
        self.idx2label: dict = {}

    def fit(self, labels):
        unique = sorted(set(labels))
        self.label2idx = {lbl: i for i, lbl in enumerate(unique)}
        self.idx2label = {i: lbl for lbl, i in self.label2idx.items()}
        return self

    def transform(self, labels) -> np.ndarray:
        return np.array(
            [self.label2idx.get(lbl, self.unknown_index) for lbl in labels],
            dtype=np.int64
        )

    def fit_transform(self, labels) -> np.ndarray:
        return self.fit(labels).transform(labels)

    def inverse_transform(self, indices) -> list:
        return [self.idx2label.get(i, '<UNKNOWN>') for i in indices]


In [9]:
priority_encoder = LabelEncoder(unknown_index=-1)
priority_encoded  = priority_encoder.fit_transform(df['Ticket Priority'].tolist())

print('label -> index mapping:', priority_encoder.label2idx)
print('first 10 encodings:', priority_encoded[:10])

assert list(priority_encoder.inverse_transform(priority_encoded[:5])) == \
       df['Ticket Priority'].tolist()[:5], 'round-trip check failed!'
print('round-trip check: pass')

unseen = priority_encoder.transform(['Extreme', 'Low'])
print('unseen category test :', unseen)

label -> index mapping: {'Critical': 0, 'High': 1, 'Low': 2, 'Medium': 3}
first 10 encodings: [0 0 2 2 2 2 0 0 2 0]
round-trip check: pass
unseen category test : [-1  2]


### One hot encoding

In [10]:
class OneHotEncoder:
    def __init__(self):
        self.category2idx: dict = {}
        self.num_classes: int   = 0

    def fit(self, categories):
        unique = sorted(set(categories))
        self.category2idx = {c: i for i, c in enumerate(unique)}
        self.num_classes   = len(unique)
        return self

    def transform(self, categories) -> np.ndarray:
        n = len(categories)
        matrix = np.zeros((n, self.num_classes), dtype=np.float32)
        for row, cat in enumerate(categories):
            idx = self.category2idx.get(cat, None)
            if idx is not None:
                matrix[row, idx] = 1.0
        return matrix

    def fit_transform(self, categories) -> np.ndarray:
        return self.fit(categories).transform(categories)

In [11]:
channel_encoder = OneHotEncoder()
channel_onehot  = channel_encoder.fit_transform(df['Ticket Channel'].tolist())

print('category -> index mapping:', channel_encoder.category2idx)
print(f'One-hot matrix shape    : {channel_onehot.shape}')
print('first 5 rows:')
print(channel_onehot[:5])


unseen_oh = channel_encoder.transform(['Telegram', 'Email'])
print('\nunseen category -> all-zeros:', (unseen_oh[0] == 0).all())

category -> index mapping: {'Chat': 0, 'Email': 1, 'Phone': 2, 'Social media': 3}
One-hot matrix shape    : (8469, 4)
first 5 rows:
[[0. 0. 0. 1.]
 [1. 0. 0. 0.]
 [0. 0. 0. 1.]
 [0. 0. 0. 1.]
 [0. 1. 0. 0.]]

unseen category -> all-zeros: True


### Sparse Representation or TF-IDF

### Tokenization

In [12]:
def tokenize(text: str) -> list:
    text = text.lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    tokens = text.split()
    return tokens

In [13]:
sample = "I'm having an issue with the GoPro Hero. Please assist!"
print('Sample tokens:', tokenize(sample))

Sample tokens: ['i', 'm', 'having', 'an', 'issue', 'with', 'the', 'gopro', 'hero', 'please', 'assist']


### N GRams

In [14]:
def generate_ngrams(tokens: list, n: int) -> list:
    return ['_'.join(tokens[i:i+n]) for i in range(len(tokens) - n + 1)]

In [15]:
    
def tokenize_with_ngrams(text: str, max_n: int = 3) -> list:
    tokens = tokenize(text)
    combined = list(tokens)
    for n in range(2, max_n + 1):
        combined.extend(generate_ngrams(tokens, n))
    return combined

In [16]:
demo_tokens = tokenize('not working is working fine')
print('Bigrams :', generate_ngrams(demo_tokens, 2))
print('Trigrams:', generate_ngrams(demo_tokens, 3))

Bigrams : ['not_working', 'working_is', 'is_working', 'working_fine']
Trigrams: ['not_working_is', 'working_is_working', 'is_working_fine']


### BOW top 5000 vocabulary

In [17]:
class CountVectorizer:
    def __init__(self, max_features: int = 5000, max_n: int = 3):
        self.max_features = max_features
        self.max_n        = max_n
        self.vocab: dict  = {}       # token → col index
        self.vocab_size: int = 0

    def fit(self, corpus: list):
        counter = Counter()
        for doc in corpus:
            tokens = tokenize_with_ngrams(doc, self.max_n)
            counter.update(tokens)
        top_tokens = [tok for tok, _ in counter.most_common(self.max_features)]
        self.vocab      = {tok: i for i, tok in enumerate(top_tokens)}
        self.vocab_size = len(self.vocab)
        return self

    def transform(self, corpus: list) -> torch.Tensor:
        rows, cols, vals = [], [], []
        for row_idx, doc in enumerate(corpus):
            tokens  = tokenize_with_ngrams(doc, self.max_n)
            cnt     = Counter(tok for tok in tokens if tok in self.vocab)
            for tok, c in cnt.items():
                rows.append(row_idx)
                cols.append(self.vocab[tok])
                vals.append(float(c))

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values  = torch.tensor(vals, dtype=torch.float32)
        return torch.sparse_coo_tensor(
            indices, values,
            size=(len(corpus), self.vocab_size)
        ).coalesce()

    def fit_transform(self, corpus: list) -> torch.Tensor:
        return self.fit(corpus).transform(corpus)

In [18]:
descriptions = df['Ticket Description'].tolist()

cv = CountVectorizer(max_features=5000, max_n=3)
bow_sparse = cv.fit_transform(descriptions)

print(f'Vocabulary size   : {cv.vocab_size}')
print(f'BoW sparse shape  : {bow_sparse.shape}')
print(f'Non-zero elements : {bow_sparse._nnz()}')

Vocabulary size   : 5000
BoW sparse shape  : torch.Size([8469, 5000])
Non-zero elements : 928769


### TF-IDF

In [19]:
class TFIDFVectorizer:
    def __init__(self, max_features: int = 5000, max_n: int = 3):
        self.cv   = CountVectorizer(max_features=max_features, max_n=max_n)
        self.idf_: torch.Tensor = None

    def _compute_idf(self, bow_sparse: torch.Tensor, N: int) -> torch.Tensor:
        bow_dense = bow_sparse.to_dense() # (N, vocab)
        df = (bow_dense > 0).sum(dim=0).float() #(vocab,)
        idf = torch.log((1 + N) / (1 + df)) + 1 #smooth
        return idf

    def fit(self, corpus: list):
        bow_sparse = self.cv.fit_transform(corpus)
        N          = len(corpus)
        self.idf_  = self._compute_idf(bow_sparse, N) #(vocab_size,)
        return self
        
    def transform(self, corpus: list) -> torch.Tensor:
        rows, cols, vals = [], [], []
        for row_idx, doc in enumerate(corpus):
            tokens   = tokenize_with_ngrams(doc, self.cv.max_n)
            doc_len  = max(len(tokens), 1)
            cnt      = Counter(tok for tok in tokens if tok in self.cv.vocab)
            for tok, c in cnt.items():
                col_idx = self.cv.vocab[tok]
                tf      = c / doc_len
                tfidf   = tf * self.idf_[col_idx].item()
                rows.append(row_idx)
                cols.append(col_idx)
                vals.append(tfidf)

        indices = torch.tensor([rows, cols], dtype=torch.long)
        values  = torch.tensor(vals, dtype=torch.float32)
        sparse  = torch.sparse_coo_tensor(
            indices, values,
            size=(len(corpus), self.cv.vocab_size)
        ).coalesce()
        return sparse

    def fit_transform(self, corpus: list) -> torch.Tensor:
        return self.fit(corpus).transform(corpus)

    def transform_single(self, text: str) -> torch.Tensor:
        tokens   = tokenize_with_ngrams(text, self.cv.max_n)
        doc_len  = max(len(tokens), 1)
        cnt      = Counter(tok for tok in tokens if tok in self.cv.vocab)
        vec      = torch.zeros(self.cv.vocab_size, dtype=torch.float32)
        for tok, c in cnt.items():
            col_idx    = self.cv.vocab[tok]
            tf         = c / doc_len
            vec[col_idx] = tf * self.idf_[col_idx].item()
        return vec

In [20]:

print('fitting TF-IDF')
tfidf_vectorizer = TFIDFVectorizer(max_features=5000, max_n=3)
tfidf_sparse     = tfidf_vectorizer.fit_transform(descriptions)

print(f'TF-IDF sparse shape  : {tfidf_sparse.shape}')
print(f'non-zero elements    : {tfidf_sparse._nnz()}')
print(f'IDF vector shape     : {tfidf_vectorizer.idf_.shape}')
print(f'IDF min / max        : {tfidf_vectorizer.idf_.min():.4f} / {tfidf_vectorizer.idf_.max():.4f}')

fitting TF-IDF
TF-IDF sparse shape  : torch.Size([8469, 5000])
non-zero elements    : 928769
IDF vector shape     : torch.Size([5000])
IDF min / max        : 1.0000 / 8.9457


### GloVe download

In [21]:
GLOVE_KAGGLE_DIR = '/kaggle/working/glove'
GLOVE_DIR  = './glove'
GLOVE_URL  = 'https://nlp.stanford.edu/data/glove.6B.zip'
GLOVE_FILE = os.path.join(GLOVE_DIR, 'glove.6B.300d.txt')
GLOVE_DIM  = 300

os.makedirs(GLOVE_DIR, exist_ok=True)

if not os.path.exists(GLOVE_FILE):
    print('downloading GloVe 6B')
    zip_path = os.path.join(GLOVE_DIR, 'glove.6B.zip')
    
    urllib.request.urlretrieve(GLOVE_URL, zip_path)
    
    print('Extracting ...')
    with zipfile.ZipFile(zip_path, 'r') as z:
        z.extract('glove.6B.300d.txt', GLOVE_DIR)
    
    os.remove(zip_path)
    print('doneing.............. Yahahahaha RRRRaaaaaaaaaa')
else:
    print('gloVe file already present.')

gloVe file already present.


In [22]:
def load_glove(filepath: str, dim: int = 300):
    word2vec = {}
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.rstrip().split(' ')
            word  = parts[0]
            vec   = np.array(parts[1:], dtype=np.float32)
            if len(vec) == dim:
                word2vec[word] = vec
    return word2vec

In [23]:
print('loading GloVe vectorz')
glove = load_glove(GLOVE_FILE, GLOVE_DIM)
print(f'Loaded {len(glove):,} GloVe vectors of dimension {GLOVE_DIM}.')

loading GloVe vectorz
Loaded 400,000 GloVe vectors of dimension 300.


### Embedding

In [24]:
PAD_TOKEN = '<PAD>'
UNK_TOKEN = '<UNK>'

glove_words   = [PAD_TOKEN, UNK_TOKEN] + list(glove.keys())
word2idx      = {w: i for i, w in enumerate(glove_words)}

vocab_size_gl = len(glove_words)
print(f'embedding vocab size : {vocab_size_gl:,}')

embedding_matrix = np.zeros((vocab_size_gl, GLOVE_DIM), dtype=np.float32)

rng = np.random.default_rng(SEED)
embedding_matrix[1] = rng.normal(scale=0.6, size=(GLOVE_DIM,))

for word, idx in word2idx.items():
    if word in glove:
        embedding_matrix[idx] = glove[word]

embedding_layer = nn.Embedding(
    num_embeddings  = vocab_size_gl,
    embedding_dim   = GLOVE_DIM,
    padding_idx     = 0
)
embedding_layer.weight.data.copy_(torch.from_numpy(embedding_matrix))
embedding_layer.weight.requires_grad = False   # frozen
embedding_layer = embedding_layer.to(device)

embedding vocab size : 400,002


In [25]:
print('embedding layer device:', next(embedding_layer.parameters()).device)
print(f'embedding weight shape: {embedding_layer.weight.shape}')

embedding layer device: cuda:0
embedding weight shape: torch.Size([400002, 300])


### TF-IDF weighting

### Convert a document into a TF-IDF-weighted mean GloVe vector.
    Steps:
      1. Tokenize (unigrams only for GloVe lookup; n-grams aren't in GloVe)
      2. Look up TF-IDF weight for each token from tfidf_vec
      3. Look up GloVe embedding; OOV → UNK vector
      4. Compute weighted mean: sum(w_i * e_i) / sum(w_i)

In [26]:
def text_to_glove_vector(
    text: str,
    tfidf_vec: TFIDFVectorizer,
    emb_layer: nn.Embedding,
    w2idx: dict,
    unk_idx: int,
    dim: int = 300
) -> torch.Tensor:

    tokens = tokenize(text)#unigrams only
    if not tokens:
        return torch.zeros(dim, device=device)

    doc_len    = len(tokens)
    tfidf_map  = Counter(tok for tok in tokens if tok in tfidf_vec.cv.vocab)
    weights    = []
    for tok in tokens:
        if tok in tfidf_vec.cv.vocab:
            col  = tfidf_vec.cv.vocab[tok]
            tf   = tfidf_map.get(tok, 0) / doc_len
            w    = tf * tfidf_vec.idf_[col].item()
        else:
            w    = 1e-6
        weights.append(max(w, 1e-9))

    weights_t  = torch.tensor(weights, dtype=torch.float32, device=device)
    
    idx_list   = [w2idx.get(tok, unk_idx) for tok in tokens]
    idx_t      = torch.tensor(idx_list, dtype=torch.long, device=device)

    with torch.no_grad():
        embeds = emb_layer(idx_t)

    weighted   = embeds * weights_t.unsqueeze(1)
    sent_vec   = weighted.sum(dim=0) / weights_t.sum()
    return sent_vec

UNK_IDX = word2idx[UNK_TOKEN]

In [27]:
test_vec = text_to_glove_vector(
    descriptions[0], tfidf_vectorizer, embedding_layer, word2idx, UNK_IDX
)
print(f'sentence vector shape : {test_vec.shape}')
print(f'vector norm           : {test_vec.norm().item():.4f}')

sentence vector shape : torch.Size([300])
vector norm           : 3.2273


# Task 1

### building all encodings

In [28]:
priority_tensor = torch.tensor(priority_encoded, dtype=torch.long)

channel_tensor  = torch.tensor(channel_onehot, dtype=torch.float32)

BATCH = 512
N     = len(descriptions)
glove_matrix = torch.zeros(N, GLOVE_DIM, dtype=torch.float32)

print(f'computing GloVe vectors for {N:,} tickets in batches of {BATCH} ...')
for start in range(0, N, BATCH):
    end   = min(start + BATCH, N)
    batch = descriptions[start:end]
    vecs  = torch.stack([
        text_to_glove_vector(doc, tfidf_vectorizer, embedding_layer, word2idx, UNK_IDX)
        for doc in batch
    ])
    glove_matrix[start:end] = vecs.cpu()
    if (start // BATCH) % 10 == 0:
        print(f'  Processed {end:,} / {N:,}')

computing GloVe vectors for 8,469 tickets in batches of 512 ...
  Processed 512 / 8,469
  Processed 5,632 / 8,469


In [29]:
print(f'\nall encodings built.')
print(f'priority_tensor : {priority_tensor.shape}')
print(f'channel_tensor  : {channel_tensor.shape}')
print(f'tfidf_sparse    : {tfidf_sparse.shape}')
print(f'glove_matrix    : {glove_matrix.shape}')

assert priority_tensor.shape[0] == channel_tensor.shape[0] == tfidf_sparse.shape[0] == glove_matrix.shape[0] == N, 'index alignment mismatch!'
print('\nindex alignment check: pass')


all encodings built.
priority_tensor : torch.Size([8469])
channel_tensor  : torch.Size([8469, 4])
tfidf_sparse    : torch.Size([8469, 5000])
glove_matrix    : torch.Size([8469, 300])

index alignment check: pass


# Task 2

### FinalScore = α·TF-IDF_score + (1-α)·GloVe_score  (α = 0.4)

In [30]:
def cosine_similarity_sparse_dense(
    query_tfidf: torch.Tensor,#dense(vocab,)
    corpus_tfidf: torch.Tensor,#sparse(N,vocab
    batch_size: int = 2048
) -> torch.Tensor:

    q   = query_tfidf.to(device).unsqueeze(0)
    q_n = q / (q.norm(dim=1, keepdim=True) + 1e-9)

    N   = corpus_tfidf.shape[0]
    sims = torch.zeros(N, device=device)

    corpus_dense = corpus_tfidf.to_dense()
    for start in range(0, N, batch_size):
        end   = min(start + batch_size, N)
        chunk = corpus_dense[start:end].to(device)
        c_n   = chunk / (chunk.norm(dim=1, keepdim=True) + 1e-9)
        sims[start:end] = (c_n @ q_n.T).squeeze(1)

    return sims

In [31]:
def cosine_similarity_dense(
    query_vec: torch.Tensor, corpus_mat: torch.Tensor, batch_size: int = 4096) -> torch.Tensor:
    q   = query_vec.to(device).unsqueeze(0)
    q_n = q / (q.norm() + 1e-9)

    N    = corpus_mat.shape[0]
    sims = torch.zeros(N, device=device)

    for start in range(0, N, batch_size):
        end   = min(start + batch_size, N)
        chunk = corpus_mat[start:end].to(device)
        c_n   = chunk / (chunk.norm(dim=1, keepdim=True) + 1e-9)
        sims[start:end] = (c_n @ q_n.T).squeeze(1)

    return sims

In [32]:
def hybrid_search(
    query: str,
    df: pd.DataFrame, tfidf_vec: TFIDFVectorizer, tfidf_corpus: torch.Tensor, glove_corpus: torch.Tensor,
    emb_layer: nn.Embedding, w2idx: dict, unk_idx: int, alpha: float = 0.4, top_k: int = 5
) -> pd.DataFrame:
    
    q_tfidf = tfidf_vec.transform_single(query)
    q_glove = text_to_glove_vector(query, tfidf_vec, emb_layer, w2idx, unk_idx)
    tfidf_scores = cosine_similarity_sparse_dense(q_tfidf, tfidf_corpus).cpu()

    glove_scores = cosine_similarity_dense(q_glove, glove_corpus).cpu()

    final_scores = alpha * tfidf_scores + (1.0 - alpha) * glove_scores

    topk_indices = torch.topk(final_scores, k=top_k).indices.numpy()
    result = df.iloc[topk_indices].copy()
    result['tfidf_score'] = tfidf_scores[topk_indices].numpy().round(4)
    result['glove_score'] = glove_scores[topk_indices].numpy().round(4)
    result['final_score'] = final_scores[topk_indices].numpy().round(4)
    result = result.sort_values('final_score', ascending=False).reset_index(drop=True)
    result.index += 1
    return result

### running queries as example

In [33]:
QUERY_1 = 'battery not charging properly after firmware update'

results_1 = hybrid_search(
    query        = QUERY_1,
    df           = df,
    tfidf_vec    = tfidf_vectorizer,
    tfidf_corpus = tfidf_sparse,
    glove_corpus = glove_matrix,
    emb_layer    = embedding_layer,
    w2idx        = word2idx,
    unk_idx     = UNK_IDX, alpha = 0.4, top_k = 5
)

display_cols = ['Ticket Subject', 'Ticket Type', 'Ticket Priority','Ticket Channel', 'tfidf_score', 'glove_score', 'final_score']

print(f"\n=== Query: '{QUERY_1}' ===")
print(results_1[display_cols].to_string())
print('\nTop-1 Description snippet:')
print(results_1['Ticket Description'].iloc[0][:300])


=== Query: 'battery not charging properly after firmware update' ===
             Ticket Subject           Ticket Type Ticket Priority Ticket Channel  tfidf_score  glove_score  final_score
1             Display issue       Technical issue          Medium           Chat       0.2987       0.6842         0.53
2            Refund request  Cancellation request             Low          Phone       0.2987       0.6842         0.53
3  Peripheral compatibility  Cancellation request        Critical          Phone       0.2987       0.6842         0.53
4            Refund request       Technical issue            High          Email       0.2987       0.6842         0.53
5      Cancellation request        Refund request          Medium          Phone       0.2987       0.6842         0.53

Top-1 Description snippet:
I'm having an issue with the {product_purchased}. Please assist. I'm using the original charger that came with my {product_purchased}, but it's not charging properly.


### TASK 3

In [34]:
import torch
import torch.nn as nn
import time

class HybridSimilarityModule(nn.Module):
    def __init__(self, tfidf_sparse, glove_matrix, alpha=0.4):
        super().__init__()
        self.register_buffer(
            'corpus_n_tfidf',
            tfidf_sparse.to_dense().float()
                .div(tfidf_sparse.to_dense().norm(dim=1, keepdim=True) + 1e-9)
        )
        self.register_buffer(
            'corpus_n_glove',
            glove_matrix.float().div(glove_matrix.norm(dim=1, keepdim=True) + 1e-9)
        )
        self.alpha = alpha

    def forward(self, queries_tfidf, queries_glove):
        q_n_tfidf = queries_tfidf / (queries_tfidf.norm(dim=1, keepdim=True) + 1e-9)
        q_n_glove = queries_glove / (queries_glove.norm(dim=1, keepdim=True) + 1e-9)
        sim_tfidf = torch.matmul(q_n_tfidf, self.corpus_n_tfidf.T)
        sim_glove = torch.matmul(q_n_glove, self.corpus_n_glove.T)
        return self.alpha * sim_tfidf + (1.0 - self.alpha) * sim_glove

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device selected: {device}")

hybrid_model = HybridSimilarityModule(tfidf_sparse, glove_matrix, alpha=0.4)

if torch.cuda.device_count() > 1:
    print(f"Multiple GPUs detected: {torch.cuda.device_count()} GPUs will be used")
    hybrid_model = nn.DataParallel(hybrid_model)

hybrid_model = hybrid_model.to(device)
hybrid_model.eval()
print("Model is ready for similarity computation.\n")


test_queries = descriptions[:100]

q_tfidf_list = []
q_glove_list = []

for q in test_queries:
    q_tfidf_list.append(tfidf_vectorizer.transform_single(q))
    q_glove_list.append(text_to_glove_vector(q, tfidf_vectorizer, embedding_layer, word2idx, UNK_IDX))

queries_tfidf_tensor = torch.stack(q_tfidf_list).to(device)
queries_glove_tensor = torch.stack(q_glove_list).to(device)

print(f"Processing batch of 100 queries...")
print(f"TF-IDF tensor shape: {queries_tfidf_tensor.shape}")
print(f"GloVe tensor shape: {queries_glove_tensor.shape}")

start_time = time.time()
with torch.no_grad():
    batch_scores = hybrid_model(queries_tfidf_tensor, queries_glove_tensor)
end_time = time.time()

print(f"Batch processing completed in {end_time - start_time:.4f} seconds")
print(f"Batch scores shape: {batch_scores.shape} -> (100 queries x {batch_scores.shape[1]} documents)")



Device selected: cuda
Model is ready for similarity computation.

Processing batch of 100 queries...
TF-IDF tensor shape: torch.Size([100, 5000])
GloVe tensor shape: torch.Size([100, 300])
Batch processing completed in 0.0258 seconds
Batch scores shape: torch.Size([100, 8469]) -> (100 queries x 8469 documents)


### App Link
**Hosted Streamlit/Gradio Space:** [Insert Link Here]
